# 2. Methodology

## Feature Engineering and Preprocessing

In [2]:
import sys
import os

# Adds the parent directory to the search path
sys.path.append(os.path.abspath(".."))

In [4]:
from src.preprocessing import (
    split_features_target,
    stratified_train_test_split,
    scale_features,
    apply_smote,
)
from src.config import RANDOM_SEED, PROCESSED_DIR
from src.logging_utils import get_logger

logger = get_logger("notebook_02")

X, y = split_features_target(df, target_col="Outcome")
X_train, X_test, y_train, y_test = stratified_train_test_split(X, y)

X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)
X_train_balanced, y_train_balanced = apply_smote(X_train_scaled, y_train)

# Save processed data
X_train_scaled.to_csv(PROCESSED_DIR / "X_train_scaled.csv", index=True)
X_test_scaled.to_csv(PROCESSED_DIR / "X_test_scaled.csv", index=True)
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=True)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=True)
X_train_balanced.to_csv(PROCESSED_DIR / "X_train_balanced.csv", index=False)
y_train_balanced.to_csv(PROCESSED_DIR / "y_train_balanced.csv", index=False)

ModuleNotFoundError: No module named 'imblearn'

## Model Selection and Hyperparameter Tuning

In [5]:
from src.modeling import (
    get_base_models,
    tune_gradient_boosting,
    tune_random_forest,
    tune_xgboost,
    cross_validate_models,
)

base_models = get_base_models()

gb_best = tune_gradient_boosting(X_train_balanced, y_train_balanced)
rf_best = tune_random_forest(X_train_balanced, y_train_balanced)
xgb_best = tune_xgboost(X_train_balanced, y_train_balanced)

# Replace base models with tuned versions
base_models["Gradient Boosting"] = gb_best
base_models["Random Forest"] = rf_best
base_models["XGBoost"] = xgb_best

cv_results = cross_validate_models(base_models, X_train_balanced, y_train_balanced)
cv_results

[2026-03-25 10:06:44] [src.modeling] [INFO] Initializing base models


NameError: name 'X_train_balanced' is not defined

## Individual Model Test Set Performance

In [6]:
from src.evaluation import evaluate_model, get_confusion_and_report, compute_roc_pr_curves
from src.visualization import plot_roc_curves

results = []
roc_curves = {}

for name, model in base_models.items():
    model.fit(X_train_balanced, y_train_balanced)
    metrics = evaluate_model(model, X_test_scaled, y_test, model_name=name)
    results.append(metrics)

    fpr, tpr, precision, recall = compute_roc_pr_curves(model, X_test_scaled, y_test)
    roc_curves[name] = {"fpr": fpr, "tpr": tpr}

results_df = pd.DataFrame(results)
results_df

NameError: name 'X_train_balanced' is not defined

In [7]:
from pathlib import Path
from src.config import FIGURES_DIR

plot_roc_curves(roc_curves, FIGURES_DIR / "roc_curves.png")|

SyntaxError: invalid syntax (447433412.py, line 4)

In [8]:
logreg_model = base_models["Logistic Regression"]
cm, report = get_confusion_and_report(
    logreg_model, X_test_scaled, y_test, target_names=["No Diabetes", "Diabetes"]
)
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", report)

NameError: name 'X_test_scaled' is not defined